<a href="https://colab.research.google.com/github/LuizaRamos/TOL506M_Final_Project/blob/main/notebooks/03_task3_zeroshot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TÖL506M - Introduction to Deep Neural Network
## Final Project: Wildlife Image Classification
### Task 3 - Zero Shot
**name:** Luiza V Sampaio Ramos, **email:** lvs2@gmail.com

In this notebook, a classification model is fine-tuned given a previous pretrained one. The model uses the dataset [Animals-10](https://www.kaggle.com/datasets/alessiocorrado99/animals10) and the data split analised in the previous notebooks([00_data_exploration.ipynb](https://github.com/LuizaRamos/TOL506M_Final_Project/blob/main/notebooks/00_data_exploration.ipynb), [01_task1_scratch.ipynb](https://github.com/LuizaRamos/TOL506M_Final_Project/blob/main/notebooks/01_task1_scratch.ipynb) and [02_task2_finetune.ipynb](https://github.com/LuizaRamos/TOL506M_Final_Project/blob/main/notebooks/02_task2_finetune.ipynb)).

**Observations:**

1.   the first six (6) cells of this notebookwere were implemented to be able to run the notebook using Google Colab mantainig it connected to the GitHub project. The rest o the code was initially wroten using DataSpell.
2.   After running the last cell, the one which impements the zero shot classification the notebook becames incopatible with GitHub.
3.   To better estimate the uncertainty three (3) notebook in total, were ran with distinct random seed. The other two notebooks are avalible at: [03_task3_zeroshot_2](https://colab.research.google.com/github/LuizaRamos/TOL506M_Final_Project/blob/main/notebooks/03_task3_zeroshot_2.ipynb) and [03_task3_zeroshot_3.ipynb](https://colab.research.google.com/github/LuizaRamos/TOL506M_Final_Project/blob/main/notebooks/03_task3_zeroshot_3.ipynb)

In [1]:
!rm -rf /content/TOL506M_Final_Project

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/LuizaRamos/TOL506M_Final_Project.git"
REPO_DIR = "/content/TOL506M_Final_Project"

# Always clone fresh in Colab
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

In [ ]:
import sys
import os
import subprocess
import importlib
import json
import time
import random
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from collections import Counter
from PIL import Image
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

project_root = Path.cwd()
if project_root.name != "TOL506M_Final_Project":
    original = project_root
    while project_root.name != "TOL506M_Final_Project" and project_root != project_root.parent:
        project_root = project_root.parent

    if project_root.name == "TOL506M_Final_Project":
        os.chdir(project_root)
        print(f"Changed working directory from {original} to {project_root}")
    else:
        raise RuntimeError(
            "Could not locate the TOL506M_Final_Project root directory. "
            "Please run this notebook/script from within the project tree."
        )
else:
    print(f"Working directory: {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Plot styling
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 12})

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Working directory: /content/TOL506M_Final_Project

PyTorch version: 2.9.0+cu126
CUDA available: True


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alessiocorrado99/animals10")

print(f'Dataset downloaded: {path}\n')

Using Colab cache for faster access to the 'animals10' dataset.
Dataset downloaded: /kaggle/input/animals10



In [ ]:
# Import project modules
from config import Config
from data.dataset import (WildlifeDataset, SplitIndices, stratified_split,
                          compute_class_counts, materialize_split, is_italian,
                          translate_names, get_class_names, get_data_loaders)
from data.augmentation import get_train_transforms, get_val_transforms
from models.resnet_scratch import ResNet18Scratch
from tasks.task3 import zero_shot_classification
from utils.training import train_epoch, validate, EarlyStopping
from utils.evaluation import evaluate_model, get_confusion_matrix, compute_metrics
from utils.visualization import plot_training_curves, plot_confusion_matrix

Config.NUM_WORKERS = 0 # avoid multiprocessing issues in Colab

data_fractions = Config.DATA_FRACTIONS
train_size = Config.TRAIN_SPLIT
val_size = Config.VAL_SPLIT
test_size = Config.TEST_SPLIT
random_seed = Config.RANDOM_SEED

data_path = Path(path) / 'raw-img'
Config.DATA_PATH = data_path

basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Download and split dataset as previously done on previous notebooks
full_dataset = datasets.ImageFolder(root=str(data_path), transform=basic_transform)
dataset_transformed = translate_names(full_dataset)

wildlife = WildlifeDataset(str(data_path), transform=basic_transform)
train_idx_base, val_idx_fixed, test_idx_fixed = stratified_split(
    full_dataset,
    train_size=train_size,
    val_size=val_size,
    test_size=test_size,
    random_seed=random_seed
)

fixed = SplitIndices(train=train_idx_base, val=val_idx_fixed, test=test_idx_fixed)

print(f'Total images: {len(wildlife)}')
print(f'Base Train images: {len(train_idx_base)}')
print(f'Validation images: {len(val_idx_fixed)}')
print(f'Test images: {len(test_idx_fixed)}')

Total images: 26179
Base Train images: 18325
Validation images: 3927
Test images: 3927


In [ ]:
# Create output dirs if they don't exist
Config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
Config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
Config.METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Task 3: Zero Shot Classification
test_dataset_zero_shot = Subset(full_dataset, test_idx_fixed)

zero_shot_test_loader = DataLoader(
    test_dataset_zero_shot,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

# Make sure we are in pure zero-shot mode
Config.SIGLIP_LINEAR_PROBE = False

zero_shot_results = zero_shot_classification(
    config=Config(),
    fixed_indices=fixed,              # optional, for consistency with other tasks
    test_loader=zero_shot_test_loader,
    train_linear_probe=False          # explicitly say: no linear probe
)

In [ ]:
train_dataset_zero_shot = Subset(full_dataset, train_idx_base)
val_dataset_zero_shot = Subset(full_dataset, val_idx_fixed)
test_dataset_zero_shot = Subset(full_dataset, test_idx_fixed)  # same as above

linear_train_loader = DataLoader(
    train_dataset_zero_shot,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=Config.NUM_WORKERS,
)

linear_val_loader = DataLoader(
    val_dataset_zero_shot,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=Config.NUM_WORKERS,
)

linear_test_loader = DataLoader(
    test_dataset_zero_shot,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=Config.NUM_WORKERS,
)

# Enable linear probe mode
Config.SIGLIP_LINEAR_PROBE = True

linear_probe_results = zero_shot_classification(
    config=Config(),
    fixed_indices=fixed,                   # same splits as Tasks 1 & 2
    train_loader=linear_train_loader,
    val_loader=linear_val_loader,
    test_loader=linear_test_loader,
    train_linear_probe=True               # turn on linear probe
)

In [ ]:
!zip -r results_task3.zip /content/TOL506M_Final_Project/results/

  adding: content/TOL506M_Final_Project/results/ (stored 0%)
  adding: content/TOL506M_Final_Project/results/plots/ (stored 0%)
  adding: content/TOL506M_Final_Project/results/metrics/ (stored 0%)
  adding: content/TOL506M_Final_Project/results/models/ (stored 0%)


In [ ]:
from google.colab import files
files.download("results_task3.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>